# MAgent2 MF-DSRQ Evaluation

Run reference-style head-to-head Battle comparisons between trained MF-DSRQ checkpoints and BenchMARL baselines. Wins use the reference kill rule, and both side assignments are evaluated.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "discrete_action_space").is_dir() and (path / "requirements.txt").exists():
            return path
    raise RuntimeError("Could not find the SRE-DQN repo root from the current notebook directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.mean_field_dsrq.notebook_utils import (
    evaluate_mfdsrq_against_baselines,
    find_latest_benchmarl_run,
    mfdsrq_epsilon_config,
    notebook_mfdsrq_config,
    plot_mfdsrq_baseline_win_rates,
)

TOTAL_STEPS = 20_000
NUM_ENVS = 2
MAP_SIZE = 40
MAX_CYCLES = 400
SEED = 42
SRE_SOLVER_WORKERS = 8
EVAL_EPISODES = 20
EVAL_MAX_STEPS = MAX_CYCLES
EPSILONS = [0.01, 0.1, 0.5, 1.0]
BASELINE_ALGORITHMS = ("mappo", "ippo", "iql")

MFDSRQ_OUTPUT_ROOT = ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "mf_dsrq_epsilon_training"
BASELINE_ROOT = ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "benchmarl_magent2_notebooks"

BASE_CFG = notebook_mfdsrq_config(
    total_steps=TOTAL_STEPS,
    num_envs=NUM_ENVS,
    map_size=MAP_SIZE,
    max_cycles=MAX_CYCLES,
    output_dir=MFDSRQ_OUTPUT_ROOT,
    seed=SEED,
    extra_overrides={"sre_solver_workers": SRE_SOLVER_WORKERS},
)


def mfdsrq_checkpoint_dir(epsilon):
    cfg = mfdsrq_epsilon_config(BASE_CFG, epsilon, MFDSRQ_OUTPUT_ROOT)
    return Path(cfg["output_dir"]) / cfg["env_name"] / f"mf_dsrq_seed{cfg['seed']}"


def compare_epsilon(epsilon):
    cfg = mfdsrq_epsilon_config(BASE_CFG, epsilon, MFDSRQ_OUTPUT_ROOT)
    checkpoint_dir = mfdsrq_checkpoint_dir(epsilon)
    comparison = evaluate_mfdsrq_against_baselines(
        cfg,
        checkpoint_dir,
        baseline_root=BASELINE_ROOT,
        algorithms=BASELINE_ALGORITHMS,
        num_episodes=EVAL_EPISODES,
        max_steps=EVAL_MAX_STEPS,
        evaluate_both_sides=True,
    )
    display(pd.DataFrame(comparison["rows"]))
    plot_mfdsrq_baseline_win_rates(comparison)
    return comparison

COMPARISONS = {}
{
    "mfdsrq_output_root": str(MFDSRQ_OUTPUT_ROOT),
    "baseline_root": str(BASELINE_ROOT),
    "algorithms": BASELINE_ALGORITHMS,
    "eval_episodes_per_side_assignment": EVAL_EPISODES,
}


## Baseline Checkpoints

In [ ]:
# Optional: inspect which baseline checkpoints will be used.
{
    algorithm: str(find_latest_benchmarl_run(algorithm, BASELINE_ROOT, task_name="battle"))
    for algorithm in BASELINE_ALGORITHMS
}


## Robust Epsilon 0.01

In [ ]:
epsilon = 0.01
comparison_eps_0_01 = compare_epsilon(epsilon)
COMPARISONS[epsilon] = comparison_eps_0_01
comparison_eps_0_01["summary"] if "summary" in comparison_eps_0_01 else comparison_eps_0_01["rows"]


## Robust Epsilon 0.1

In [ ]:
epsilon = 0.1
comparison_eps_0_1 = compare_epsilon(epsilon)
COMPARISONS[epsilon] = comparison_eps_0_1
comparison_eps_0_1["summary"] if "summary" in comparison_eps_0_1 else comparison_eps_0_1["rows"]


## Robust Epsilon 0.5

In [ ]:
epsilon = 0.5
comparison_eps_0_5 = compare_epsilon(epsilon)
COMPARISONS[epsilon] = comparison_eps_0_5
comparison_eps_0_5["summary"] if "summary" in comparison_eps_0_5 else comparison_eps_0_5["rows"]


## Robust Epsilon 1.0

In [ ]:
epsilon = 1.0
comparison_eps_1_0 = compare_epsilon(epsilon)
COMPARISONS[epsilon] = comparison_eps_1_0
comparison_eps_1_0["summary"] if "summary" in comparison_eps_1_0 else comparison_eps_1_0["rows"]


## Aggregate Table

In [ ]:
all_rows = []
for epsilon, comparison in COMPARISONS.items():
    for row in comparison["rows"]:
        all_rows.append({"epsilon": epsilon, **row})
pd.DataFrame(all_rows)
